# Download Data

Sample up to 1000 documents per source from the [`onyx-dot-app/EnterpriseRAG-Bench`](https://huggingface.co/datasets/onyx-dot-app/EnterpriseRAG-Bench) Hugging Face dataset and save each as raw JSON under `sources/<source>/` locally. Nothing is uploaded to S3 at this stage — that happens after cleaning, in `3_upload_to_s3.ipynb`. Safe to re-run if interrupted (e.g. lost internet): already-saved files are skipped. Run this before `2_clean_data.ipynb`.

In [4]:
import os
import json


sources = ["confluence", "fireflies", "github", "gmail", "google_drive", "hubspot", "jira", "linear", "slack"]


## Download 1000 files per source from the Hugging Face benchmark dataset

Pull the `documents` split of [`onyx-dot-app/EnterpriseRAG-Bench`](https://huggingface.co/datasets/onyx-dot-app/EnterpriseRAG-Bench) (one row per document, columns `doc_id` / `source_type` / `title` / `content`), sample up to 1000 documents per source, and save each as raw JSON under `sources/<source>/`. Cleaning into `.txt` and uploading to S3/the KB prefixes happens next, in `2_clean_data.ipynb` and `3_upload_to_s3.ipynb`.

The full dataset is highly imbalanced per source (Slack ~275,000, Gmail ~120,000, Linear ~35,000, Google Drive ~25,000, HubSpot ~15,000, Fireflies ~10,000, GitHub ~8,000, Jira ~6,000, Confluence ~5,000). After inspecting these counts, only a random sample of up to 1000 documents per source is used, so no single source dominates the resulting knowledge base.

Requires `HF_TOKEN` in `data_pipeline/.env` and the `huggingface_hub`, `pandas`, and `pyarrow` packages.

In [5]:
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()

hf_dataset = "onyx-dot-app/EnterpriseRAG-Bench"
hf_sample_size = 1000


# Download the EnterpriseRAG-Bench documents parquet and load it into a DataFrame.
def download_hf_documents():

    path = hf_hub_download(repo_id = hf_dataset, repo_type = "dataset", filename = "data/documents/test.parquet", token = os.environ.get("HF_TOKEN"))

    return pd.read_parquet(path)


# Sample up to `sample_size` documents for one source and save each as raw JSON. Already-saved
# files are skipped, so re-running after a lost connection just picks up where it left off.
def process_hf_source(documents_df, source, sample_size = hf_sample_size):

    local_folder = f"sources/{source}"
    os.makedirs(local_folder, exist_ok = True)

    source_df = documents_df[documents_df["source_type"] == source]
    size = min(sample_size, len(source_df))
    sampled = source_df.sample(n = size, random_state = 42)

    skipped = 0

    for i, row in enumerate(sampled.itertuples(), 1):

        local_path = os.path.join(local_folder, f"{row.doc_id}.json")

        if os.path.exists(local_path):
            skipped += 1
            continue

        doc = {"doc_id": row.doc_id, "title": row.title, "content": row.content}

        with open(local_path, "w", encoding = "utf-8") as f:
            json.dump(doc, f, ensure_ascii = False, indent = 2)

        if i % 100 == 0:
            print(f"{source}: processed {i}/{size}")

    print(f"{source}: total available={len(source_df)}, saved={size - skipped} new, skipped={skipped} already present in {local_folder}")


In [6]:
documents_df = download_hf_documents()

for source in sources:
    process_hf_source(documents_df, source)


confluence: total available=5189, saved=0 new, skipped=1000 already present in sources/confluence
fireflies: total available=10173, saved=0 new, skipped=1000 already present in sources/fireflies
github: total available=8052, saved=0 new, skipped=1000 already present in sources/github
gmail: total available=121390, saved=0 new, skipped=1000 already present in sources/gmail
google_drive: total available=25108, saved=0 new, skipped=1000 already present in sources/google_drive
hubspot: total available=15017, saved=0 new, skipped=1000 already present in sources/hubspot
jira: total available=6120, saved=0 new, skipped=1000 already present in sources/jira
linear: total available=35308, saved=0 new, skipped=1000 already present in sources/linear
slack: total available=285605, saved=0 new, skipped=1000 already present in sources/slack
